In [ ]:
import os
import sys
from dotenv import load_dotenv

# 1. Path Fix & Auto-Reload
# This ensures that if you edit your .py files, the notebook sees the changes
%load_ext autoreload
%autoreload 2

# Move up to the project root so we can see the 'app' folder
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. Load Environment Variables
# Explicitly point to the .env file in the app directory
dotenv_path = os.path.join(root_path, 'app', '.env')
load_dotenv(dotenv_path)

# 3. Imports (Now they will find the database config correctly)
from app.database.connection import SessionLocal
from app.database.repository import Repository
from app.database.models import ScrapedArticle
    

# 4. Initialize Database Session
db = SessionLocal()
repo = Repository(db)

# 5. Fetch and Process
# Using get_all_scraped_articles (ensure this is in your repository.py)
articles = repo.get_all_scraped_articles(limit=5)
for art in articles:
            print(f"\nProcessing: {art.article_name}")
            # This is where you will test your Regex patterns
            raw_text = art.full_content
            print(f"Raw Length: {len(raw_text)} chars")
    
db.close() # Always close the session in a notebook to prevent connection leaks

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

Processing: Societal Impacts
Raw Length: 4810 chars

Processing: How Tolan builds voice-first AI with GPT-5.1
Raw Length: 9220 chars

Processing: Sharing our compliance framework for California's Transparency in Frontier AI Act
Raw Length: 6403 chars


In [ ]:
articles = repo.get_all_scraped_articles(limit=5)

In [27]:
import os
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [4]:
import re

def clean_anthropic_example(text: str) -> str:
    # 1. REMOVE HEADER NOISE (Everything up to the main section)
    # We look for "Societal Impacts" as the starting anchor
    start_marker = "Societal Impacts"
    if start_marker in text:
        # We find the FIRST occurrence and keep everything after
        text = text[text.find(start_marker):]

    # 2. REMOVE FOOTER NOISE (Everything after the main content)
    # The content usually ends before the "Join the Research team" or "Publications" search
    end_markers = ["Join the Research team", "Publications\nSearch", "Products\nClaude"]
    for marker in end_markers:
        if marker in text:
            text = text.split(marker)[0]
            break

    # 3. CLEAN INTERNAL BOILERPLATE
    # Remove repetitive UI fragments often found in Anthropic/OpenAI pages
    ui_noise = [
        r"Skip to main content",
        r"Skip to footer",
        r"Try Claude",
        r"Back to Overview",
        r"Customize\s+cookie settings.*", # Removes the cookie block at the end
        r"Reject\s+all cookies.*",
        r"Accept\s+all cookies.*"
    ]
    for pattern in ui_noise:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE | re.DOTALL)

    # 4. NORMALIZE WHITESPACE
    # Replace 3+ newlines with 2 (paragraphs)
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove leading/trailing spaces on each line
    text = "\n".join([line.strip() for line in text.split('\n')])
    
    return text.strip()

# --- Execution in Notebook ---
# Assuming 'art' is your ScrapedArticle object
cleaned_content = clean_anthropic_example(art.full_content)

print("=== CLEANED OUTPUT ===")
print(cleaned_content[:1000]) # Print first 1000 chars to verify

=== CLEANED OUTPUT ===
Research
Economic Futures
Commitments
Learn
News

Policy
Sharing our compliance framework for California's Transparency in Frontier AI Act
Dec 20, 2025

On January 1, California's Transparency in Frontier AI Act (SB 53) will go into effect. It establishes the nation’s first frontier AI safety and transparency requirements for catastrophic risks.

While we have long advocated for a federal framework, Anthropic endorsed SB 53 because we believe frontier AI developers like ourselves should be transparent about how they assess and manage these risks. Importantly, the law balances the need for strong safety practices, incident reporting, and whistleblower protections—while preserving flexibility in how developers implement their safety measures, and exempting smaller companies from unnecessary regulatory burdens.

One of the law’s key requirements is that frontier AI developers publish a framework describing how they assess and manage catastrophic risks. Our Frontier 

In [7]:
articles = repo.get_all_scraped_articles()
for art in articles:
            print(f"\nProcessing: {art.article_name}")
            # This is where you will test your Regex patterns
            raw_text = art.full_content
            print(f"Raw Length: {len(raw_text)} chars")
            cleaned_content = clean_anthropic_example(art.full_content)
            print("=== CLEANED OUTPUT ===")
            print(cleaned_content[:100]) # Print first 1000 chars to verify


Processing: Societal Impacts
Raw Length: 4810 chars
=== CLEANED OUTPUT ===
Societal Impacts

Working closely with the Anthropic Policy and Safeguards teams, Societal Impacts i

Processing: How Tolan builds voice-first AI with GPT-5.1
Raw Length: 9220 chars
=== CLEANED OUTPUT ===
Log in
Switch to
ChatGPT
(opens in a new window)
Sora
(opens in a new window)
API Platform
(opens in

Processing: Sharing our compliance framework for California's Transparency in Frontier AI Act
Raw Length: 6403 chars
=== CLEANED OUTPUT ===
Research
Economic Futures
Commitments
Learn
News

Policy
Sharing our compliance framework for Califo


In [11]:
import trafilatura

def clean_article(raw_html):
    # Extract main content, excluding comments and navigation noise
    # 'favor_precision' ensures we don't accidentally grab sidebar 'Most Popular' text
    cleaned_text = trafilatura.extract(raw_html, 
                                       include_comments=False, 
                                       include_tables=True,
                                       no_fallback=False,
                                       favor_precision=True)
    return cleaned_text

# Example usage with your TechCrunch content:
# clean_content = clean_article(scraped_techcrunch_html)

In [12]:
import re

def clean_techcrunch_by_anchors(text: str) -> str:
    # 1. FIND THE START (Skip the menu/navigation)
    # We look for a date pattern (e.g., January 9, 2026) 
    # and take everything after it.
    date_pattern = r'[A-Z][a-z]+ \d{1,2}, 202[0-9]'
    match = re.search(date_pattern, text)
    if match:
        # Move start to the end of the date string
        text = text[match.end():].strip()

    # 2. FIND THE END (Cut off the footer/sidebar)
    # TechCrunch usually lists "Topics" at the bottom of the content.
    if "Topics" in text:
        # We split by 'Topics' and keep the first half
        text = text.split("Topics")[0]

    # 3. POLISH: Remove 'IMAGE CREDITS' and social callouts
    # Often found right at the start or end of the sliced content
    text = re.sub(r'IMAGE CREDITS:.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Join the Disrupt.*?WAITLIST NOW', '', text, flags=re.DOTALL)

    return text.strip()

In [21]:
# utils/cleaners.py
import trafilatura

def engine_clean(html_content, site_type):
    # Layer 1: Heuristic (Universal)
    content = trafilatura.extract(html_content)
    
    # Layer 2: Rule-Based (Site Specific)
    if site_type == "moneycontrol":
        content = content.split("READ MORE")[0]
    elif site_type == "techcrunch":
        content = content.split("Topics")[0]
        
    return content

In [22]:
# 1. Fetch a small sample from your repository
sample_articles = repo.get_all_scraped_articles()[:5] 

print(f"--- TESTING CLEANING ON {len(sample_articles)} ARTICLES ---\n")

for art in sample_articles:
    # We use the raw 'full_content' currently in your DB
    raw_text = art.full_content
    
    # Run the cleaner
    # Note: Pass art.source_type so your anchor-logic knows which site it's on
    cleaned_text = engine_clean(raw_text, art.source_type)
    
    print(f"TITLE: {art.article_name}")
    print(f"SOURCE: {art.source_type}")
    print(f"ORIGINAL LENGTH: {len(raw_text)} chars")
    print(f"CLEANED LENGTH: {len(cleaned_text)} chars")
    print("-" * 20)
    
    # Preview the first 500 chars of the cleaned version
    print("CLEANED PREVIEW:")
    print(cleaned_text[:500] + "...")
    print("\n" + "="*50 + "\n")

--- TESTING CLEANING ON 5 ARTICLES ---

TITLE: Buy HDFC Bank; target of Rs 1,850: ICICI Securities
SOURCE: general
ORIGINAL LENGTH: 4865 chars


TypeError: object of type 'NoneType' has no len()

In [30]:
import operator
from typing import Annotated, List, TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- 1. State Definition ---
class ChunkState(TypedDict):
    """State for a single parallel worker."""
    chunk: str

class OverallState(TypedDict):
    """The global state of the graph."""
    full_content: str
    # 'operator.add' allows parallel workers to append to this list 
    # without overwriting each other.
    valid_summaries: Annotated[list, operator.add] 
    final_report: str

# --- 2. Node Functions ---

async def chunking_node(state: OverallState):
    """Splits the article and decides which chunks to send to workers."""
    text = state["full_content"]
    splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
    chunks = splitter.split_text(text)
    
    # We return a list of 'Send' objects. 
    # LangGraph will run 'gatekeeper_node' for each one in parallel.
    return [Send("gatekeeper", {"chunk": c}) for c in chunks]

async def gatekeeper_node(state: ChunkState):
    """The AI Gatekeeper: Filters noise and summarizes signal."""
    prompt = f"""
    Analyze this web content chunk.
    If it is core news, research, or financial data, summarize it in 2 sentences.
    If it is junk (menus, ads, social media links, boilerplate), reply ONLY with 'SKIP'.
    
    CONTENT: {state['chunk']}
    """
    response = await llm.ainvoke(prompt)
    
    if "SKIP" in response.content.upper():
        return {"valid_summaries": []} # Send nothing back
        
    return {"valid_summaries": [response.content]}

async def aggregator_node(state: OverallState):
    """Combines all parallel summaries into one clean article."""
    combined = "\n\n".join(state["valid_summaries"])
    
    # Optional: Final pass to make it flow better
    final_prompt = f"Synthesize these points into a cohesive executive summary:\n\n{combined}"
    final_resp = await llm.ainvoke(final_prompt)
    
    return {"final_report": final_resp.content}

# --- 3. Graph Construction ---

builder = StateGraph(OverallState)

builder.add_node("chunker", chunking_node)
builder.add_node("gatekeeper", gatekeeper_node)
builder.add_node("aggregator", aggregator_node)

# START -> chunker -> (Parallel Workers) -> aggregator -> END
builder.add_conditional_edges(START, chunking_node, ["gatekeeper"])
builder.add_edge("gatekeeper", "aggregator")
builder.add_edge("aggregator", END)

workflow = builder.compile()

In [31]:
# 1. Grab 5 samples from your database (read-only)
sample_articles = repo.get_all_scraped_articles()[:5]

print(f"🚀 Starting Map-Reduce Test on {len(sample_articles)} articles...\n")

for i, art in enumerate(sample_articles):
    print(f"--- Article {i+1}: {art.article_name[:60]} ---")
    
    # Define the input for your workflow
    initial_state = {
        "full_content": art.full_content,
        "valid_summaries": [] # Initialize the list
    }
    
    # 2. Run the workflow (awaiting directly in Jupyter)
    result = await workflow.ainvoke(initial_state)
    
    # 3. View the outputs
    print(f"Total Chunks Processed: {len(result.get('valid_summaries', []))}")
    print("\nFINAL EXECUTIVE SUMMARY:")
    print(result["final_report"])
    print("\n" + "="*80 + "\n")

🚀 Starting Map-Reduce Test on 5 articles...

--- Article 1: Buy HDFC Bank; target of Rs 1,850: ICICI Securities ---
Total Chunks Processed: 3

FINAL EXECUTIVE SUMMARY:
Here is a cohesive executive summary:

ICICI Securities has issued a buy rating for HDFC Bank, with a target price of Rs 1,850, citing the bank's strong post-merger performance and growth prospects. The brokerage firm expects HDFC Bank to deliver an 18% CAGR in deposits and a 13% CAGR in loan growth for FY24-26, driven by a balanced loan-to-deposit ratio and borrowing substitution. Although the bank's Q4FY24 net interest margin (NIM) outcome was slightly better than expected, growth estimates have been revised downward, resulting in a 5% reduction in EPS estimates for FY25/26. Nevertheless, ICICI Securities has upgraded the stock to a "buy" rating, driven by the recent sharp correction in stock price and anticipated gradual re-rating, with the target price remaining unchanged at Rs 1,850. This suggests a potential upside

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jzjh78ste34vkq2eprrw3nyd` service tier `on_demand` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 2s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'requests', 'code': 'rate_limit_exceeded'}}